In [ ]:
generate_queries

ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant that generates multiple search queries based on a single input query.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='Generate multiple search queries related to: {question} \n OUTPUT (4 queries):'), additional_kwargs={})])
| RunnableWithFallbacks(runnable=ChatHuggingFace(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.3.18'}}, llm=HuggingFacePipeline(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.3.18'}}, pipeline=TextGenerationPipeline: {'model': 'Qwen3ForCausalLM', 'dtype': 'bfloat16', 'device': 'cuda', 'input_modalities': 'text', 'output_modalities': ('text',)}, model_id='Qwen/Qwen3-4B-Instruct-250

In [ ]:
generate_queries

ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant that generates multiple search queries based on a single input query.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='Generate multiple search queries related to: {question} \n OUTPUT (4 queries):'), additional_kwargs={})])
| RunnableWithFallbacks(runnable=ChatHuggingFace(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.3.18'}}, llm=HuggingFacePipeline(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.3.18'}}, pipeline=TextGenerationPipeline: {'model': 'Qwen3ForCausalLM', 'dtype': 'bfloat16', 'device': 'cuda', 'input_modalities': 'text', 'output_modalities': ('text',)}, model_id='Qwen/Qwen3-4B-Instruct-250

In [ ]:
queries = generate_queries.invoke({"question": original_query})
for query in queries:
    print(query)

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1. Universal Studios Singapore ticket prices and booking options
2. Universal Studios Singapore attractions and ride reviews
3. Universal Studios Singapore near me and directions
4. Universal Studios Singapore holiday packages and family travel tips


# RAG Fusion — Reciprocal Rank Fusion for Better Retrieval

### by Mayank Chugh | Advanced RAG Series (2026 Edition) | Episode 5

This notebook builds a small RAG pipeline over a personal travel corpus (Singapore notes),
then upgrades it with **RAG Fusion**: instead of retrieving once per user query, an LLM
generates several phrasings of the question, each phrasing is retrieved separately, and the
resulting ranked lists are combined with **Reciprocal Rank Fusion (RRF)**.

> 📄 Reference: Cormack, Clarke & Büttcher, *"Reciprocal Rank Fusion Outperforms Condorcet
> and Individual Rank Learning Methods,"* SIGIR '09. RRF formula and k=60 default follow
> this paper exactly.

**Stack:** Qwen3-4B-Instruct-2507, local + 4-bit (LLM, primary) → Claude Sonnet 5 (fallback) → GPT-5.6-sol (final fallback) · BAAI/bge-small-en-v1.5 (embeddings) · Chroma (vector store)
**Runtime:** T4 GPU required — the primary model now runs locally, not just as a fallback

> 🔁 The LLM is a three-way fallback chain, primary-local this time: a locally-loaded
> Qwen3-4B-Instruct-2507 model (4-bit quantized) is tried first, with no API key and no billing risk.
> If it errors — OOM, CUDA failure, etc. — LangChain automatically retries with
> Anthropic, then OpenAI if that also fails. Note that `with_fallbacks()` only reacts
> to raised exceptions, not answer quality: a successful-but-weak local response is
> still returned as-is, it won't automatically escalate to a paid model for a better
> answer.

> ✅ **This notebook was executed end to end on a Colab T4 GPU (7 Sep 2026).** Primary LLM: local Qwen3-4B-Instruct-2507, 4-bit; corpus: 6 `.txt` files mounted from Google Drive → 33 chunks. Every output below is real captured output from that run — live generated query variants, real RRF-fused retrieval, real answers. The local model answered every call, so neither paid fallback was invoked.


<a href="https://colab.research.google.com/github/mayankchugh-learning/Generative-AI-Indepth-Basic-to-Advance/blob/main/RAG_Fusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Install Dependencies

In [ ]:
!pip -q install langchain langchain-core langchain-community \
    langchain-text-splitters langchain-chroma langchain-huggingface \
    langchain-anthropic langchain-openai
!pip -q install chromadb sentence_transformers
!pip -q install transformers accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 

## 2. Utility: Text Wrapping for Readable Output

In [ ]:
import textwrap

def wrap_text(text, width=90):
    lines = text.split('\n')
    wrapped_lines = [textwrap.fill(line, width=width) for line in lines]
    return '\n'.join(wrapped_lines)

## 3. Configure the LLM — Local Qwen → Anthropic → OpenAI Fallback Chain

In [ ]:
import os
from google.colab import userdata

# The primary model runs locally and needs no key. Anthropic and OpenAI are
# only called if the local model errors — but their keys still need to be set
# upfront, since with_fallbacks() can't prompt for credentials mid-chain.
os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
import torch

# Primary: local open-weight model - no API key, no billing, runs on the Colab GPU.
# 4-bit quantization is required here, not optional: it keeps the weights small
# enough to leave room for the embedding model and the KV cache on a 16GB T4.
# Qwen3-4B-Instruct-2507 is the *non-thinking* instruct variant - it answers
# directly instead of spending the whole token budget inside a <think> block.
local_model_name = "Qwen/Qwen3-4B-Instruct-2507"

# compute dtype must be float16 on a T4: Turing has no native bfloat16 support.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(local_model_name)
model = AutoModelForCausalLM.from_pretrained(
    local_model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

local_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=False,        # only this flag - temperature/top_k/top_p are silently ignored on Qwen3 pipelines
    return_full_text=False, # return the completion only, not prompt + completion
)

primary_llm = ChatHuggingFace(
    llm=HuggingFacePipeline(pipeline=local_pipeline),
    tokenizer=tokenizer,
)

# Fallback 1: Anthropic Claude - used automatically if the local model errors (OOM,
# CUDA failure, etc.)
fallback_llm_1 = ChatAnthropic(model="claude-sonnet-5")

# Fallback 2: OpenAI - used if both the local model and Anthropic fail
fallback_llm_2 = ChatOpenAI(model="gpt-5.6-sol")

# Chain them: try the local model first, then Anthropic, then OpenAI
llm = primary_llm.with_fallbacks([fallback_llm_1, fallback_llm_2])

result = llm.invoke("Write a ballad about LangChain")
print(result.content)

# Note: with_fallbacks() only triggers on a raised exception (OOM, timeout, API
# error) - it has no concept of answer quality. If the local model runs successfully
# but gives a weak answer, that answer is returned as-is; it will not silently retry
# with Claude or GPT just because the response could have been better.

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress

**Ballad of LangChain**  
*(A Lighthearted, Rhyming Tale of AI and Code)*  

In a world of code and dreams, where logic flows like streams,  
There stood a name that sparked the AI scene —  
**LangChain**, bold and wise, with purpose clear,  
A bridge between the prompt and the machine’s sphere.  

No need to build from scratch, no need to start anew,  
With LangChain, you can guide your model through.  
From prompts to plans, from data to action,  
It helps your AI think, and act, and *conquer* the action.  

It weaves a chain — a smart, well-tuned chain —  
Of tools, prompts, and memory, strong and plain.  
With agents that roam, and retrievers that find,  
It pulls the truth from databases, deep and wide.  

It speaks to LLMs, like a wise old friend,  
“Here’s your task, my friend — go find the end.”  
With memory buffers, and stateful flow,  
It remembers what you said, and what you know.  

You ask a question, “Who won the 2024 race?”  
LangChain pulls the data, checks the base.  

## 4. Core Imports

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
import langchain  # used later for langchain.debug

## 5. Load the Corpus

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data_path = "/content/drive/MyDrive/English"

In [ ]:
%%time
loader = DirectoryLoader(
    data_path,
    glob="*.txt",
    loader_cls=TextLoader,
    show_progress=True,
)
docs = loader.load()

100%|██████████| 6/6 [00:06<00:00,  1.07s/it]

CPU times: user 16.8 ms, sys: 5.86 ms, total: 22.7 ms
Wall time: 8.08 s


In [ ]:
len(docs)

6

In [ ]:
docs = docs[:50]
len(docs)

6

In [ ]:
docs[0]

Document(metadata={'source': '/content/drive/MyDrive/English/singapore_transport.txt'}, page_content="Singapore Transport Notes\n\nThe MRT (subway) is the easiest way to get around and covers most of the tourist areas directly. Get an EZ-Link card or just tap a contactless credit card directly at the gates - Singapore's system supports both now, so a physical transit card isn't strictly necessary anymore if your bank card supports contactless.\n\nBuses fill in the gaps where the MRT doesn't reach, and the same EZ-Link/contactless system works across both. Google Maps handles Singapore transit routing well, including bus numbers and real-time arrival estimates, so there's rarely a need for a separate transit app.\n\nGrab (the regional Uber equivalent) is the standard ride-hailing app here and works better than trying to hail a taxi on the street in most areas. Taxis are still common and can be flagged down, but surge-like surcharges apply at night and during peak hours regardless of whe

In [ ]:
print(docs[2].page_content)

Singapore Attractions Notes

Gardens by the Bay is the one attraction that shows up on every list, and it earns the spot. The Supertree Grove is free to walk through during the day, and the nightly light show (Garden Rhapsody) is also free - two shows a night, check the schedule since times shift. The paid Cloud Forest and Flower Dome conservatories are worth the ticket price, especially Cloud Forest with its indoor waterfall.

Marina Bay Sands is technically a hotel/casino/mall complex, but the SkyPark observation deck on top is a separate ticketed attraction with the best skyline view in the city. The infinity pool is hotel-guests-only, a common point of confusion for visitors who think they can buy a day pass to swim in it - you can't, only stay guests have pool access.

Sentosa Island covers a lot of ground beyond just Universal Studios - S.E.A. Aquarium, the cable car over from Mount Faber, several beaches (Siloso, Palawan, Tanjong), and Fort Siloso if you're into WWII history. It

In [ ]:
print(docs[1].page_content)

Singapore Culture Notes

Singapore's cultural identity is genuinely a blend rather than a single dominant culture layered with minorities - Chinese, Malay, Indian, and Peranakan (a mixed Chinese-Malay heritage) traditions all have real, visible presence rather than being folded into a generic "Asian" identity. This shows up most obviously in the food and in the four official languages (English, Mandarin, Malay, Tamil).

Chinatown is not just a historic district but still has an active temple culture - the Buddha Tooth Relic Temple is the most visited, elaborate architecture and worth going inside rather than just photographing the exterior. Thian Hock Keng Temple is older and quieter, a good contrast if you want to see a working temple without the tour groups.

Little India centers around Sri Veeramakaliamman Temple and the shophouses along Serangoon Road. Deepavali (Diwali) season lights up the whole district with decorations, worth timing a visit around if your trip dates allow it.



In [ ]:
raw_text = "".join(doc.page_content for doc in docs if doc.page_content)

In [ ]:
print(raw_text)

Singapore Transport Notes

The MRT (subway) is the easiest way to get around and covers most of the tourist areas directly. Get an EZ-Link card or just tap a contactless credit card directly at the gates - Singapore's system supports both now, so a physical transit card isn't strictly necessary anymore if your bank card supports contactless.

Buses fill in the gaps where the MRT doesn't reach, and the same EZ-Link/contactless system works across both. Google Maps handles Singapore transit routing well, including bus numbers and real-time arrival estimates, so there's rarely a need for a separate transit app.

Grab (the regional Uber equivalent) is the standard ride-hailing app here and works better than trying to hail a taxi on the street in most areas. Taxis are still common and can be flagged down, but surge-like surcharges apply at night and during peak hours regardless of whether you use Grab or a traditional taxi.

Getting to Sentosa specifically has a few options: the Sentosa Exp

## 6. Chunk the Corpus

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len,
    is_separator_regex=False,
)

In [ ]:
texts = text_splitter.split_text(raw_text)

In [ ]:
len(texts)

33

In [ ]:
print(texts[4])

Walking is genuinely viable in the central areas - Marina Bay, Chinatown, and the Civic District are all connected by covered walkways in parts, which matters given the heat and frequent rain. That said, the humidity means even short walks can be tiring; budget more recovery time between activities than you would in a drier climate.
Singapore Culture Notes


## 7. Embeddings (BGE)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
model_name = "BAAI/bge-small-en-v1.5"
encode_kwargs = {'normalize_embeddings': True}  # normalize for cosine similarity

In [ ]:
embedding_function = HuggingFaceEmbeddings(
    model_name=model_name,
    #model_kwargs={'device': 'cuda'},
    encode_kwargs=encode_kwargs,
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 8. Build the Vector Store

In [ ]:
%%time
### Build the Chroma store and persist to disk
db = Chroma.from_texts(texts, embedding_function, persist_directory="./chroma_db")

CPU times: user 289 ms, sys: 100 ms, total: 389 ms
Wall time: 854 ms


In [ ]:
query = "Tell me about Universal Studios Singapore?"

db.similarity_search(query, k=5)

[Document(id='2f14ed0f-630d-42c7-b190-4dc666b7756d', metadata={}, page_content="Universal Studios Singapore is on Sentosa Island, part of the larger Resorts World Sentosa complex. It's the only Universal Studios park in Southeast Asia, and it's noticeably smaller than the Orlando or Hollywood parks - you can realistically do it in one full day without rushing."),
 Document(id='d8544951-d808-4aa2-84ba-c1f0938173a2', metadata={}, page_content='One thing to know: Singapore has strict laws about public drunkenness and noise after certain hours, and drinking on the MRT is banned (not just discouraged - actually banned, with fines). Bars generally close by 2-3am on weekdays, later on weekends in licensed zones like Clarke Quay and Orchard Towers.\nUniversal Studios Singapore Notes'),
 Document(id='4bc11c77-4063-42d1-b60d-16e27054f427', metadata={}, page_content="Singapore's nightlife scene is more varied than people expect for such a small country. Clarke Quay is the most touristy strip - ri

## 9. Set Up a Retriever

In [ ]:
retriever = db.as_retriever()  # can add mmr fetch_k=20, search_type="mmr"

retriever.invoke(query)

[Document(id='2f14ed0f-630d-42c7-b190-4dc666b7756d', metadata={}, page_content="Universal Studios Singapore is on Sentosa Island, part of the larger Resorts World Sentosa complex. It's the only Universal Studios park in Southeast Asia, and it's noticeably smaller than the Orlando or Hollywood parks - you can realistically do it in one full day without rushing."),
 Document(id='d8544951-d808-4aa2-84ba-c1f0938173a2', metadata={}, page_content='One thing to know: Singapore has strict laws about public drunkenness and noise after certain hours, and drinking on the MRT is banned (not just discouraged - actually banned, with fines). Bars generally close by 2-3am on weekdays, later on weekends in licensed zones like Clarke Quay and Orchard Towers.\nUniversal Studios Singapore Notes'),
 Document(id='4bc11c77-4063-42d1-b60d-16e27054f427', metadata={}, page_content="Singapore's nightlife scene is more varied than people expect for such a small country. Clarke Quay is the most touristy strip - ri

## 10. Baseline Chat Chain (Single Query, No Fusion)

In [ ]:
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

In [ ]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

In [ ]:
prompt = ChatPromptTemplate.from_template(template)

In [ ]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n{context}\n\nQuestion: {question}\n'), additional_kwargs={})])

In [ ]:
llm

RunnableWithFallbacks(runnable=ChatHuggingFace(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.3.18'}}, llm=HuggingFacePipeline(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.3.18'}}, pipeline=TextGenerationPipeline: {'model': 'Qwen3ForCausalLM', 'dtype': 'bfloat16', 'device': 'cuda', 'input_modalities': 'text', 'output_modalities': ('text',)}, model_id='Qwen/Qwen3-4B-Instruct-2507'), tokenizer=Qwen2Tokenizer(name_or_path='Qwen/Qwen3-4B-Instruct-2507', vocab_size=151643, model_max_length=1010000, padding_side='left', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False

In [ ]:
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
text_reply = chain.invoke("Tell me about Universal Studio Singapore")

print(wrap_text(text_reply))

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Universal Studios Singapore is located on Sentosa Island, within the larger Resorts World
Sentosa complex. It is the only Universal Studios park in Southeast Asia and is notably
smaller than the parks in Orlando or Hollywood. Visitors can typically complete a full day
at the park without needing to rush.


## 11. RAG Fusion — Generate Multiple Query Variants

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import (
    ChatPromptTemplate,
    PromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)

In [ ]:
prompt = ChatPromptTemplate(
    input_variables=['question'],
    messages=[
        SystemMessagePromptTemplate(
            prompt=PromptTemplate(
                input_variables=[],
                template='You are a helpful assistant that generates multiple search queries based on a single input query.'
            )
        ),
        HumanMessagePromptTemplate(
            prompt=PromptTemplate(
                input_variables=['question'],
                template='Generate multiple search queries related to: {question} \n OUTPUT (4 queries):'
            )
        ),
    ]
)

In [ ]:
original_query = "universal studios Singapore"

In [ ]:
generate_queries = (
    prompt
    | llm
    | StrOutputParser()
    | (lambda x: [line.strip() for line in x.split("\n") if line.strip()])
)

In [ ]:
generate_queries

ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant that generates multiple search queries based on a single input query.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='Generate multiple search queries related to: {question} \n OUTPUT (4 queries):'), additional_kwargs={})])
| RunnableWithFallbacks(runnable=ChatHuggingFace(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.3.18'}}, llm=HuggingFacePipeline(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.3.18'}}, pipeline=TextGenerationPipeline: {'model': 'Qwen3ForCausalLM', 'dtype': 'bfloat16', 'device': 'cuda', 'input_modalities': 'text', 'output_modalities': ('text',)}, model_id='Qwen/Qwen3-4B-Instruct-250

## 12. Reciprocal Rank Fusion

In [ ]:
from langchain_core.load import dumps, loads


def reciprocal_rank_fusion(results: list[list], k=60):
    # RRFscore(d) = sum over rankings r of 1 / (k + rank_r(d))
    # Cormack, Clarke & Buttcher, SIGIR'09
    fused_scores = {}
    for docs in results:
        # Assumes the docs are returned in sorted order of relevance
        for rank, doc in enumerate(docs):
            doc_str = dumps(doc)
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            previous_score = fused_scores[doc_str]
            fused_scores[doc_str] += 1 / (rank + k)

    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]
    return reranked_results

In [ ]:
ragfusion_chain = generate_queries | retriever.map() | reciprocal_rank_fusion

In [ ]:
langchain.debug = True

In [ ]:
ragfusion_chain.input_schema.model_json_schema()

{'properties': {'question': {'title': 'Question', 'type': 'string'}},
 'required': ['question'],
 'title': 'PromptInput',
 'type': 'object'}

In [ ]:
ragfusion_chain.invoke({"question": original_query})

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/tmp/ipykernel_2211/1029628038.py:18: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  (loads(doc), score)
/tmp/ipykernel_2211/1029628038.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


[(Document(id='2f14ed0f-630d-42c7-b190-4dc666b7756d', metadata={}, page_content="Universal Studios Singapore is on Sentosa Island, part of the larger Resorts World Sentosa complex. It's the only Universal Studios park in Southeast Asia, and it's noticeably smaller than the Orlando or Hollywood parks - you can realistically do it in one full day without rushing."),
  0.06666666666666667),
 (Document(id='407841e0-2c25-43ee-b74e-3d70dcb32edf', metadata={}, page_content="Sentosa Island covers a lot of ground beyond just Universal Studios - S.E.A. Aquarium, the cable car over from Mount Faber, several beaches (Siloso, Palawan, Tanjong), and Fort Siloso if you're into WWII history. It's connected by a short causeway or the Sentosa Express monorail from VivoCity."),
  0.04839549075403121),
 (Document(id='6b43e3d1-6831-4b06-a60c-be8a139781c2', metadata={}, page_content="Singapore Transport Notes\n\nThe MRT (subway) is the easiest way to get around and covers most of the tourist areas directly.

## 13. Full RAG Fusion Chain (End to End)

In [ ]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

# itemgetter("question") - not RunnablePassthrough() - pulls the string out of the
# input dict. RunnablePassthrough() would hand the whole {"question": ...} dict to
# the prompt, so the rendered prompt would read: Question: {'question': '...'}
full_rag_fusion_chain = (
    {
        "context": ragfusion_chain,
        "question": itemgetter("question"),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
full_rag_fusion_chain.input_schema.model_json_schema()

{'properties': {'question': {'title': 'Question'}},
 'required': ['question'],
 'title': 'RunnableParallel<context,question>Input',
 'type': 'object'}

In [ ]:
fusion_reply = full_rag_fusion_chain.invoke(
    {"question": "Tell me about Singapore's nightlife scene?"}
)

print(wrap_text(fusion_reply))

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/tmp/ipykernel_2211/1029628038.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Singapore's nightlife scene is diverse and offers a range of experiences, from vibrant
bars and clubs to cultural shows and relaxed, low-key spots. Clarke Quay is the most
touristy area, featuring riverside bars and clubs like Zouk, which host EDM and hip-hop
nights on weekends. Cover charges can range from S$20 to S$40 depending on the night and
whether a guest DJ is performing.

Bars in Chinatown and Tanjong Pagar are more low-key, offering craft beer, wine bars, and
speakeasies such as Employees Only and Jigger & Pony, which focus on proper cocktails.
Most bars close by 2–3 a.m. on weekdays, with later closing times on weekends in licensed
zones like Clarke Quay and Orchard Towers.

Singapore has strict laws regarding public drunkenness and noise after certain hours, and
drinking on the MRT is actually banned with fines. For those interested in cultural
experiences, the Wings of Time light and water show at Sentosa is a well-regarded option,
and Marina Bay Sands offers a free nightl

## 14. Sample Output

Real output captured from the live run above — local Qwen3-4B-Instruct-2507, RRF over the generated query variants:

> Singapore's nightlife scene is more varied than expected for a small country. Clarke Quay is the most touristy area, featuring riverside bars and clubs like Zouk, which attract a younger crowd with EDM and hip-hop nights on weekends. Cover charges can range from S$20 to S$40 depending on the night and whether a guest DJ is performing.
>
> Bars in Chinatown and Tanjong Pagar offer a more low-key experience, with craft beer places, wine bars, and speakeasies hidden behind unmarked doors. Well-known speakeasy-style spots include Employees Only and Jigger & Pony, which focus on proper cocktails rather than bottle service.
>
> It's important to note that Singapore has strict laws regarding public drunkenness and noise after certain hours, and drinking on the MRT is actually banned with fines. Most bars close by 2-3 a.m. on weekdays, with later closing times on weekends in licensed zones like Clarke Quay and Orchard Towers.
>
> For cultural experiences, options like the Wings of Time light and water show at Sentosa and the free nightly light show (Spectra) at Marina Bay Sands offer unique "night out" activities. These are especially worth attending if you are already in the area.

Note what the fusion bought: nightlife venues, the alcohol and noise laws, and the Sentosa / Marina Bay light shows came from different chunks of the corpus, surfaced by different phrasings of the same question and merged by RRF into one context.


## Resources

- **Paper:** Cormack, Clarke & Büttcher, ["Reciprocal Rank Fusion Outperforms Condorcet and Individual Rank Learning Methods"](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf), SIGIR '09
- **LangChain fallbacks:** [`Runnable.with_fallbacks()` reference](https://python.langchain.com/docs/how_to/fallbacks/)
- **Model docs (verify current model strings before running — these change fast):**
  - [Anthropic API models](https://docs.claude.com/en/docs/about-claude/models)
  - [OpenAI API models](https://platform.openai.com/docs/models)
  - [Qwen3 on Hugging Face](https://huggingface.co/Qwen)
- **Notebook and full code:** [GitHub — mayankchugh-learning](https://github.com/mayankchugh-learning/Generative-AI-Indepth-Basic-to-Advance)
- **Channel:** [YouTube — @itaienthusiast](https://www.youtube.com/@itaienthusiast) · [Medium — @mayankchugh.jobathk](https://medium.com/@mayankchugh.jobathk) · [mayankchugh-learning.github.io](https://mayankchugh-learning.github.io)

Next episode: **FlashRank** — a lightweight, code-only reranking module.
